# ZWatch - Team V

A fresh system anomaly detection prototype for IBM Z Datathon 2026.

[Open the deployed dashboard](http://148.100.79.194:38900) and enter the deployment access token. Source: https://github.com/Hemanthk101/Team-V-ZWatch-LinuxONE

Live telemetry and synthetic replays use separate models. All figures below are from the saved held-out synthetic evaluation; they do not establish production accuracy.

In [1]:
import json, platform
from pathlib import Path
import pandas as pd
print("Current notebook architecture:", platform.machine())
report = Path("data/evaluation.json")
if not report.exists():
    report = Path("evidence/evaluation.json")
evidence = json.loads(report.read_text())
evidence["runtime"]

Current notebook architecture: s390x


{'architecture': 's390x',
 'python': '3.12.15',
 'logical_cpus': 3,
 'memory_gib': 7.82,
 'measured_at_utc': '2026-10-17T14:47:37.207+00:00',
 'measurement_scope': 'Python inference inside constrained ZWatch LinuxONE container'}

## Repeatable evaluation

800 training observations (seed 17), 256 calibration observations (seed 31), and six independent 180-sample evaluation scenarios (seed 4001). Each incident needs three consecutive abnormal samples to open and three clear samples to close.

In [2]:
pd.DataFrame(evidence["results"])[["title","detected_events","expected_events","guardrail_detected_events","precision","recall","false_positive_rate","inference_p95_ms"]]

,title,detected_events,expected_events,guardrail_detected_events,precision,recall,false_positive_rate,inference_p95_ms
0,Cascading service degradation,2,2,1,0.9385,0.9385,0.0348,6.788
1,CPU saturation,2,2,2,0.9385,0.9385,0.0348,6.758
2,Memory pressure,2,2,2,0.9385,0.9385,0.0348,6.751
3,Network surge,2,2,2,0.9385,0.9385,0.0348,6.762
4,Storage and I/O pressure,2,2,2,0.9385,0.9385,0.0348,6.618
5,Healthy workload control,0,0,0,0.0000,0.0000,0.0000,6.668


## Architecture and limits

The psutil collector, Isolation Forest, API, and SQLite store all run on LinuxONE CPU. No external model service is called. Explanations describe positive deviations from the baseline rather than a proven cause. Longer real baselines, TLS/SSO and availability testing are required before production deployment.

See README.md and docs/architecture.svg for details.